BERT Base Models (ABSA)

Tujuan: melampaui baseline Sprint 3 menggunakan pre-trained BERT.
- NER CRF baseline  : entity-level F1 = **0.8408**
- ABSA SVM baseline : macro-F1 = **0.7669** | multi-conflict acc = **0.6156**

In [2]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from collections import Counter
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForSequenceClassification,
    DataCollatorForTokenClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from seqeval.metrics import classification_report as seq_report, f1_score as seq_f1
from sklearn.metrics import classification_report, f1_score

PROJECT_ROOT   = Path('..').resolve()
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR      = PROJECT_ROOT / 'backend' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device :', DEVICE)
if DEVICE == 'cuda':
    print('GPU    :', torch.cuda.get_device_name(0))
print('Data   :', DATA_PROCESSED)
print('Models :', MODEL_DIR)

/home/anton/TextMining/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cuda
GPU    : NVIDIA GeForce RTX 4070 Laptop GPU
Data   : /home/anton/data/processed
Models : /home/anton/backend/models


## ABSA dengan BERT-base-uncased

### Load data ABSA

In [9]:
absa_train = pd.read_csv(DATA_PROCESSED / 'absa_train.csv')
absa_val   = pd.read_csv(DATA_PROCESSED / 'absa_val.csv')
absa_test  = pd.read_csv(DATA_PROCESSED / 'absa_test.csv')

print(f'train: {len(absa_train)} pairs | val: {len(absa_val)} | test: {len(absa_test)}')
print('Label dist (train):', dict(Counter(absa_train['label'])))
print('Contoh input:')
print(absa_train[['title', 'entity', 'label']].head(3).to_string(index=False))

train: 11433 pairs | val: 1430 | test: 1448
Label dist (train): {1: 4341, 2: 4048, 0: 3044}
Contoh input:
                                                   title        entity  label
                       MMTC Q2 net loss at Rs 10.4 crore          MMTC      1
       Mid-cap funds can deliver more, stay put: Experts Mid-cap funds      2
Market seeing patience, if not conviction: Prakash Diwan        Market      1


### Tokenisasi & Dataset

Input dikirim sebagai sentence-pair: title dan entity dipisah oleh [SEP] secara otomatis oleh tokenizer.  
BERT menggunakan token_type_ids untuk membedakan segmen sehingga persis yang diperlukan agar model membedakan  
konteks judul dari entitas target.

Mengapa ini lebih baik dari BoW: BERT menghasilkan representasi kontekstual contoh token "Bank" pada  
"YES Bank" memiliki embedding berbeda dari "Bank" pada "bank merger". BoW tidak punya kapasitas ini.

In [10]:
ABSA_MODEL_NAME = 'bert-base-uncased'
absa_tokenizer  = AutoTokenizer.from_pretrained(ABSA_MODEL_NAME)

class ABSADataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.encodings = tokenizer(
            df['title'].tolist(),
            df['entity'].tolist(),
            truncation=True,
            max_length=max_length,
            padding=False,
        )
        self.labels = df['label'].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

absa_train_ds = ABSADataset(absa_train, absa_tokenizer)
absa_val_ds   = ABSADataset(absa_val,   absa_tokenizer)
absa_test_ds  = ABSADataset(absa_test,  absa_tokenizer)
print('Dataset sizes — train:', len(absa_train_ds),
      '| val:', len(absa_val_ds),
      '| test:', len(absa_test_ds))

Dataset sizes — train: 11433 | val: 1430 | test: 1448


### Fine-tune BERT ABSA

In [11]:
absa_model = AutoModelForSequenceClassification.from_pretrained(
    ABSA_MODEL_NAME,
    num_labels=3,
    id2label={0: 'negative', 1: 'neutral', 2: 'positive'},
    label2id={'negative': 0, 'neutral': 1, 'positive': 2},
)

def compute_absa_metrics(p):
    logits, labels = p
    pred_ids = np.argmax(logits, axis=1)
    return {'macro_f1': f1_score(labels, pred_ids, average='macro')}

absa_args = TrainingArguments(
    output_dir=str(MODEL_DIR / 'absa_bert_checkpoints'),
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=100,
    fp16=(DEVICE == 'cuda'),
    report_to='none',
)

absa_trainer = Trainer(
    model=absa_model,
    args=absa_args,
    train_dataset=absa_train_ds,
    eval_dataset=absa_val_ds,
    data_collator=DataCollatorWithPadding(absa_tokenizer),
    compute_metrics=compute_absa_metrics,
    processing_class=absa_tokenizer,
)

print('Training ABSA BERT...')
absa_trainer.train()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2055.51it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Training ABSA BERT...


Epoch,Training Loss,Validation Loss,Macro F1
1,0.477333,0.413789,0.851685
2,0.299314,0.394048,0.863026
3,0.173971,0.444235,0.866641
4,0.132761,0.490487,0.867311


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.33it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

TrainOutput(global_step=1432, training_loss=0.2986322704640181, metrics={'train_runtime': 98.8141, 'train_samples_per_second': 462.808, 'train_steps_per_second': 14.492, 'total_flos': 767152392004656.0, 'train_loss': 0.2986322704640181, 'epoch': 4.0})

### Evaluasi dengan per-class F1 & perbandingan baseline

In [12]:
absa_pred_out = absa_trainer.predict(absa_test_ds)
absa_pred_ids = np.argmax(absa_pred_out.predictions, axis=1)
true_labels   = absa_test['label'].tolist()

names = ['negative', 'neutral', 'positive']
print('=== ABSA BERT-base-uncased – TEST set ===')
print(classification_report(true_labels, absa_pred_ids, target_names=names, digits=4))

bert_absa_f1  = f1_score(true_labels, absa_pred_ids, average='macro')
svm_macro_f1  = 0.7669
print(f'Macro-F1 (BERT): {bert_absa_f1:.4f}')
print(f'Macro-F1 (SVM) : {svm_macro_f1:.4f}')
print(f'Delta          : {bert_absa_f1 - svm_macro_f1:+.4f}')

=== ABSA BERT-base-uncased – TEST set ===
              precision    recall  f1-score   support

    negative     0.8848    0.8802    0.8825       384
     neutral     0.8544    0.8777    0.8659       548
    positive     0.8986    0.8760    0.8871       516

    accuracy                         0.8778      1448
   macro avg     0.8793    0.8780    0.8785      1448
weighted avg     0.8782    0.8778    0.8779      1448

Macro-F1 (BERT): 0.8785
Macro-F1 (SVM) : 0.7669
Delta          : +0.1116


### Key Analisis dengan single vs multi-conflict

Ini adalah tes utama: apakah BERT menutup gap akurasi untuk headline dengan  
**entitas berkonflik sentimen** (masalah utama BoW di Sprint 3)?

In [13]:
absa_test_eval = absa_test.copy()
absa_test_eval['pred']    = absa_pred_ids
absa_test_eval['correct'] = (absa_test_eval['pred'] == absa_test_eval['label'])

grp    = absa_test_eval.groupby('s_no')
n_ent  = grp['entity'].transform('size')
n_sent = grp['sentiment'].transform('nunique')

def categorize(ne, ns):
    if ne == 1:   return 'single'
    return 'multi-conflict' if ns > 1 else 'multi-uniform'

absa_test_eval['head_type'] = [categorize(ne, ns) for ne, ns in zip(n_ent, n_sent)]

summary = absa_test_eval.groupby('head_type').agg(
    n_pairs  = ('correct', 'size'),
    bert_acc = ('correct', 'mean'),
).round(4)

summary['svm_acc'] = {'multi-conflict': 0.6156, 'multi-uniform': 0.7819, 'single': 0.8213}
summary['delta']   = (summary['bert_acc'] - summary['svm_acc']).round(4)

print('Perbandingan akurasi per tipe headline (BERT vs SVM baseline):')
print(summary.to_string())
print()
print(f'Gap single vs multi-conflict (BERT) : '
      f"{summary.loc['single','bert_acc'] - summary.loc['multi-conflict','bert_acc']:.4f}")
print(f'Gap single vs multi-conflict (SVM)  : 0.2057')

Perbandingan akurasi per tipe headline (BERT vs SVM baseline):
                n_pairs  bert_acc  svm_acc   delta
head_type                                         
multi-conflict      294    0.8503   0.6156  0.2347
multi-uniform       376    0.8537   0.7819  0.0718
single              778    0.8997   0.8213  0.0784

Gap single vs multi-conflict (BERT) : 0.0494
Gap single vs multi-conflict (SVM)  : 0.2057


In [14]:
# Contoh kasus multi-conflict — apakah BERT memperbaikinya?
print('Contoh multi-conflict (5 headline):')
err_or_all = absa_test_eval[absa_test_eval['head_type'] == 'multi-conflict']
id2name = {0: 'neg', 1: 'neu', 2: 'pos'}
shown = 0
for sno in err_or_all['s_no'].unique()[:6]:
    g = absa_test_eval[absa_test_eval['s_no'] == sno]
    if g['correct'].all():
        continue  # tampilkan hanya yang ada kesalahan
    print('  TITLE:', g['title'].iloc[0])
    for _, r in g.iterrows():
        mark = 'OK ' if r['correct'] else 'XX '
        print(f'     {mark} {r["entity"]:25s} gold={id2name[r["label"]]} pred={id2name[r["pred"]]}')
    print()
    shown += 1
    if shown >= 5:
        break

Contoh multi-conflict (5 headline):
  TITLE: Trade long on infrastructure stocks: Devang Visaria, Devangvisaria.com
     XX  infrastructure stocks     gold=pos pred=neu
     OK  Devangvisaria.com         gold=neu pred=neu



In [15]:
absa_save_path = MODEL_DIR / 'absa_bert_base_uncased'
absa_model.save_pretrained(absa_save_path)
absa_tokenizer.save_pretrained(absa_save_path)
print('Saved ABSA model to:', absa_save_path)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.48it/s]

Saved ABSA model to: /home/anton/backend/models/absa_bert_base_uncased


In [12]:
#model size

def dir_size_mb(path):
    return round(sum(f.stat().st_size for f in path.rglob('*') if f.is_file()) / 1e6, 2)

print('NER BERT model size (MB):', dir_size_mb(MODEL_DIR / 'ner_bert_base_cased'))
print('ABSA BERT Cased model size (MB):', dir_size_mb(MODEL_DIR / 'absa_bert_base_uncased'))

NER BERT model size (MB): 431.58
ABSA BERT Cased model size (MB): 438.67
